Notes perso :
page 848 - 860

Partially based on some older recalls i have of an MCMC course I had in M1 (for stat physique I believe)



# MFCS Miniproject 3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Implementation steps

GBM model function for trajectory simulation

In [ ]:

# Physics like, T/dt

def simulate_gbm(x0=1, mu=1.0, sigma=1.0, T=1.0, dt=0.01):
    N = int(T/dt) # Steps
    t = np.linspace(0, T, N) # discrete
    x = np.zeros(N)
    x[0] = x0 # init
    Z = np.random.standard_normal(N - 1)

    # Actual sim
    for i in range(N - 1):
        # Math here
        x[i+1] = x[i] * np.exp((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z[i])
    return t, x

### Log-likelyhood

In [ ]:
def gbm_log_likelihood(data, dt, mu, sigma):
    # log(Xi+1) - log(Xi)
    log_increments = np.diff(np.log(data))


    # GBM follows normal distribution
    # Get the proba distrib function => need mean + var
    mean_inc = (mu - 0.5 * sigma**2) * dt
    var_inc = (sigma**2) * dt
    
    n = len(log_increments)
    sum_sq_diff = np.sum((log_increments - mean_inc)**2)
    
    # Plug the normal distrib in a ln gives that :
    log_lik = -0.5 * (n * np.log(2 * np.pi * var_inc) + (sum_sq_diff / var_inc))
    
    return log_lik

GBM trajectory test, log scale

In [ ]:

for _ in range(500):
    t, x = simulate_gbm()
    plt.plot(t, x, alpha=0.05, color='blue')
plt.xlabel('Time')
plt.yscale('log')
plt.ylabel('GBM')
plt.show()

#### Move set function(s)

In [ ]:

# random walk
def random_walk(mu, sigma, distrib_sigma=0.1):

    # new = current + offset + noise
    new_mu = np.random.normal(loc=mu, scale=distrib_sigma)
    new_sigma = np.random.normal(loc=sigma, scale=distrib_sigma)
        
    return new_mu, new_sigma

#### Metropolis-Hasting criterion

Based on the pseudo-algo in the notes

In [ ]:

def accept_move(L_new, L_old):
    log_alpha = L_new - L_old
    
    return np.log(np.random.uniform(0, 1)) < log_alpha

### Mandatory Task 1

My main contact with MCMC was through parameter optimization, and the classic statistical computation of pi using uniformly dropped points in a square. I'm also aware that MCMC is used for prameter-space exploration for mathematical models (or is it a variant?).

#### Main MCMC loop

In [ ]:


def mcmc(data, dt, n_iterations, start_mu, start_sigma, step_size):
    # numpy prealloc
    samples_mu = np.zeros(n_iterations)
    samples_sigma = np.zeros(n_iterations)
    samples_log_likelihood = np.zeros(n_iterations)
    
    # init
    curr_mu = start_mu
    curr_sigma = start_sigma
    L_old = gbm_log_likelihood(data, dt, curr_mu, curr_sigma)
    
    accepted_count = 0

    # MCMC loop
    for i in range(n_iterations):
        # Compute new parameter set
        prop_mu, prop_sigma = random_walk(curr_mu, curr_sigma, step_size)
        
        # log likelyhood scoring
        L_new = gbm_log_likelihood(data, dt, prop_mu, prop_sigma)
        
        # do we accept or not?
        if accept_move(L_new, L_old):
            curr_mu = prop_mu
            curr_sigma = prop_sigma
            L_old = L_new
            accepted_count += 1
            
        # store move
        samples_mu[i] = curr_mu
        samples_sigma[i] = curr_sigma
        samples_log_likelihood[i] = L_old
        
    print(f"Acceptance Rate: {accepted_count / n_iterations:.2%}")
    return samples_mu, samples_sigma, samples_log_likelihood

In [ ]:
# --- Data from our model ---
true_mu = 0.2
true_sigma = 0.3
T = 10.0
dt = 0.01
t, data = simulate_gbm(x0=1, mu=true_mu, sigma=true_sigma, T=T, dt=dt)

# --- Run MCMC ---
mu_chain, sigma_chain, log_likelihood_chain = mcmc(data, dt, 
                                                   n_iterations=100000, 
                                                   start_mu=0.5, 
                                                   start_sigma=0.5, 
                                                   step_size=0.02)

Some datascience plots 

In [ ]:
plt.hist(mu_chain, bins=50, density=True, alpha=0.7)
plt.title("Posterior Distribution of Mu")
plt.show()

In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(log_likelihood_chain[1000:], label="Log-likelihood evolution")
plt.legend()
plt.title("Log likelihood Trace Plot")
plt.show()

In [ ]:
skip = 0
bins = 50
plt.figure(figsize=(10, 8))
plt.plot(mu_chain[skip:], sigma_chain[skip:], color='black', alpha=1.0, linewidth=1.0, label='MCMC Path', zorder=1)
# plt.plot(mu_chain[skip:], sigma_chain[skip:], color='black', alpha=1.0, linewidth=0.01, label='MCMC Path', zorder=1)
clean_mu = mu_chain[skip:]
clean_sigma = sigma_chain[skip:]
counts, xedges, yedges = np.histogram2d(clean_mu, clean_sigma, bins=bins)
max_idx = np.unravel_index(np.argmax(counts, axis=None), counts.shape)
max_mu = (xedges[max_idx[0]] + xedges[max_idx[0] + 1]) / 2
max_sigma = (yedges[max_idx[1]] + yedges[max_idx[1] + 1]) / 2
plt.xlabel('($\mu$)')
plt.ylabel('($\sigma$)')
plt.title('MCMC Parameter Space Evolution & Posterior Density')
plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


In [ ]:
# Drop the initial moves
clean_mu = mu_chain[skip:]
clean_sigma = sigma_chain[skip:]

# Value estimaion
mu_est = np.mean(clean_mu)
sigma_est = np.mean(clean_sigma)

# Error bounds
mu_std = np.std(clean_mu)
sigma_std = np.std(clean_sigma)

print(f"Mu estimate: {mu_est:.4f} ± {mu_std:.4f}")
print(f"Sigma estimate: {sigma_est:.4f} ± {sigma_std:.4f}")

### Symmetry on $\sigma$

In [ ]:
# --- Data from our model ---
true_mu = 0.2
true_sigma = 0.3
T = 10.0
dt = 0.01
t, data = simulate_gbm(x0=1, mu=true_mu, sigma=true_sigma, T=T, dt=dt)

# --- Run MCMC ---
mu_chain, sigma_chain, log_likelihood_chain = mcmc(data, dt, 
                                                   n_iterations=100000, 
                                                   start_mu=-0.5, 
                                                   start_sigma=-0.5, 
                                                   step_size=0.02)

In [ ]:
# Drop the initial moves
clean_mu = mu_chain[skip:]
clean_sigma = sigma_chain[skip:]

# Value estimaion
mu_est = np.mean(clean_mu)
sigma_est = np.mean(clean_sigma)

# Error bounds
mu_std = np.std(clean_mu)
sigma_std = np.std(clean_sigma)

print(f"Mu estimate: {mu_est:.4f} ± {mu_std:.4f}")
print(f"Sigma estimate: {sigma_est:.4f} ± {sigma_std:.4f}")

By symmetry, $-\sigma$ is a valid solution as well. Starting in the symmetrical position with $\sigma=-0.5$ will converge to $-0.3$, which is also a solution. This is due to the square in the formulas.

### Sweep on dt

In [ ]:

true_mu = 0.2
true_sigma = 0.3
T = 10.0
n_iterations = 50000
burn_in = 500

# Sweep parameters
dt_values = np.logspace(-1, -4, 25)
mu_results = []
sigma_results = []
mu_errors = []
sigma_errors = []

print("Starting sweep...")

for dt in dt_values:
    print(f"Running MCMC with step size dt={dt:.4f}...  [{(dt_values == dt).nonzero()[0][0]+1}/{len(dt_values)}]")
    t, data = simulate_gbm(x0=1, mu=true_mu, sigma=true_sigma, T=T, dt=dt)

    m_chain, s_chain, _ = mcmc(data, dt, 
                                n_iterations=n_iterations, 
                                start_mu=0.5, 
                                start_sigma=0.5, 
                                step_size=0.02)

    # Calculate estimates (Mean) and Error Bounds (Std Dev) post burn-in
    m_est = np.mean(m_chain[burn_in:])
    s_est = np.mean(s_chain[burn_in:])
    m_err = np.std(m_chain[burn_in:])
    s_err = np.std(s_chain[burn_in:])
    
    mu_results.append(m_est)
    sigma_results.append(s_est)
    mu_errors.append(m_err)
    sigma_errors.append(s_err)

# Convert to arrays for math
mu_results = np.array(mu_results)
sigma_results = np.array(sigma_results)
mu_errors = np.array(mu_errors)
sigma_errors = np.array(sigma_errors)

# Calculate Deltas (Absolute difference from ground truth)
delta_mu = np.abs(mu_results - true_mu)
delta_sigma = np.abs(sigma_results - true_sigma)



In [ ]:
# --- Plotting ---
fig, axs = plt.subplots(1, 2, figsize=(15, 5))

# 1. Mu Plot: Computed vs True
axs[0].fill_between(dt_values, mu_results - mu_errors, mu_results + mu_errors, color='blue', alpha=0.2, label='Error Bound (1 std)')
axs[0].plot(dt_values, mu_results, color='blue', label='Computed $\mu$')
axs[0].axhline(true_mu, color='red', linestyle='--', label='True $\mu$')
axs[0].set_title('Computed $\mu$ vs dt')
axs[0].invert_xaxis() # $1.0 \to 0.001$
axs[0].set_xscale('log')
axs[0].set_yscale('log')
axs[0].legend()

# 2. Sigma Plot: Computed vs True
axs[1].fill_between(dt_values, sigma_results - sigma_errors, sigma_results + sigma_errors, color='green', alpha=0.2, label='Error Bound (1 std)')
axs[1].plot(dt_values, sigma_results, color='green', label='Computed $\sigma$')
axs[1].axhline(true_sigma, color='red', linestyle='--', label='True $\sigma$')
axs[1].set_title('Computed $\sigma$ vs dt')
axs[1].invert_xaxis()
axs[1].set_xscale('log')
axs[1].set_yscale('log')
axs[1].legend()

for ax in axs.flat:
    ax.set_xlabel('Step Size (dt)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

When sweeping on dt, the precision on the "volatility" of the data $\sigma$ converges

### Sweep on T + adaptive dt

In [ ]:

true_mu = 0.2
true_sigma = 0.3
n_iterations = 50000
burn_in = 500

# Sweep parameters
T_values = np.logspace(1, 3.2, 25) #3.2 because i wanna go past 3.0, and 4 doesn't have any accepted moves, pain...
mu_results = []
sigma_results = []
mu_errors = []
sigma_errors = []

print("Starting sweep...")

for T in T_values:

    dt = T / 10000

    print(f"Running MCMC with step size T={T:.4f}...  [{(T_values == T).nonzero()[0][0]+1}/{len(T_values)}]")
    t, data = simulate_gbm(x0=1, mu=true_mu, sigma=true_sigma, T=T, dt=dt)

    m_chain, s_chain, _ = mcmc(data, dt, 
                                n_iterations=n_iterations, 
                                start_mu=0.5, 
                                start_sigma=0.5, 
                                step_size=0.02)

    # Calculate estimates (Mean) and Error Bounds (Std Dev) post burn-in
    m_est = np.mean(m_chain[burn_in:])
    s_est = np.mean(s_chain[burn_in:])
    m_err = np.std(m_chain[burn_in:])
    s_err = np.std(s_chain[burn_in:])
    
    mu_results.append(m_est)
    sigma_results.append(s_est)
    mu_errors.append(m_err)
    sigma_errors.append(s_err)

# Convert to arrays for math
mu_results = np.array(mu_results)
sigma_results = np.array(sigma_results)
mu_errors = np.array(mu_errors)
sigma_errors = np.array(sigma_errors)

# Calculate Deltas (Absolute difference from ground truth)
delta_mu = np.abs(mu_results - true_mu)
delta_sigma = np.abs(sigma_results - true_sigma)



In [ ]:
# --- Plotting ---
fig, axs = plt.subplots(1, 2, figsize=(15, 5))

# 1. Mu Plot: Computed vs True
axs[0].fill_between(T_values, mu_results - mu_errors, mu_results + mu_errors, color='blue', alpha=0.2, label='Error Bound (1 std)')
axs[0].plot(T_values, mu_results, color='blue', label='Computed $\mu$')
axs[0].axhline(true_mu, color='red', linestyle='--', label='True $\mu$')
axs[0].set_title('Computed $\mu$ vs T')
axs[0].set_xscale('log')
axs[0].set_yscale('log')
axs[0].legend()

# 2. Sigma Plot: Computed vs True
axs[1].fill_between(T_values, sigma_results - sigma_errors, sigma_results + sigma_errors, color='green', alpha=0.2, label='Error Bound (1 std)')
axs[1].plot(T_values, sigma_results, color='green', label='Computed $\sigma$')
axs[1].axhline(true_sigma, color='red', linestyle='--', label='True $\sigma$')
axs[1].set_title('Computed $\sigma$ vs T')
axs[1].set_xscale('log')
axs[1].set_yscale('log')
axs[1].legend()

for ax in axs.flat:
    ax.set_xlabel('Time window (T)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

When sweeping on T, the precision on the "drift" of the data $\mu$ converges. Note that to keep the amount of samples identical, we compute dt according to T.

#### Comment on the results

We conclude that the precision on $\sigma$ is dependent on the number of samples, which makes sense since it models the volatility of the data, therefore, the sample amount is the right quantity to vary. On the other hand, the drift $\mu$ cannot be accurately extrapolated from noisy data in a short timewindow. It is thus necessary to increase the sampling time rather than the amount of samples.

Note that when sweeping we start above 40% of accepted moves, and that we end up to under 10% of accepted moves, which may be under the threshold for statistical significancy, and could explain the relative inaccurate results when tuning $\mu$. We chose to leave that result here as it highlights the dilemma between accuracy and computation time.

### Task 1 specific answers

#### MCMC ingredients

The MCMC implemented here is composed of the following core parts :

- A Geometric Brownian Motion SDE
    - I got it from the miniproject PDF. As for the parameters, I took some random numbers that shouldn't cause explosions and keep the data in an OK state from a computational point of view. There is no magic reason for these numbers.

- A proposal distribution function
    - Having some experience with Gaussian-related statistics, I decided to go with a random walk weighted using a normal distribution which spread was in the same magnitude range of the parameters, but inferior to them, thus 0.1.

- A log-likelihood function for the GBM
    - I got it by taking the log of a normal distribution since GBMs follow a normal distribution. $\mu$ and $\sigma$ are tuned inside the MCMC loop, meaning that I made sure to not use parameters from the known sampled function when running the MCMC.

- An acceptance criterion
    - I follow the Metropolis Algorithm, according to the pseudo-code from the course notes.


#### Impact of $T$ and $dt$

As highlighted in the plots above, when capturing properties that are evolving slowly in time, it is necessary to adapt the time window accordingly, like we showed for $\mu$ in our example. In the same manner, properties dependent on the nature of the "chaos" of the data, the amount of data point is to be tuned, as we've outlined with $\sigma$


#### Estimator

To estimate the properties, we use the mean and standard deviation as a value estimate and error bound. Note that when plotting the trajectory in the parameter space, there is a startup time for the algorithm where it is guided towards a specific part of the parameter space. In order to improve accuracy, we get rid of the points corresponding to that startup time. 

#### 4 fold increase in data

As seen in the previous 2 plots and subsequent analysis, we can infer that having 4 time the amount of data will increase the precision on $\sigma$, which as previously stated represents the volatility of the data. There won't be changes on the $\mu$ precision as it is time-window dependent.


### Task 2 : Test on financial data, Bitcoin

Applying a stationary SDE model to Bitcoin data provides a rigorous test of the MCMC's limitations. Financial markets are essentially 'moving targets'; the act of characterizing a price series often leads to its own obsolescence. This phenomenon, known as market reflexivity, suggests that any parameters found today may be rendered invalid tomorrow as participants react to the same data.

In [ ]:

# Toggle to not redownload data. This parts require Yahoos's yfinance package. Please email me if you wish to run the program and I forgot to include the data file (427kb)
if False:
    import yfinance as yf

    filename="btc_1h_data.txt"
    data = yf.download("BTC-USD", period="720d", interval="1h")
    prices = data['Close'].dropna().values
    np.savetxt(filename, prices)
    print(f"Successfully saved {len(prices)} data points to {filename}")

In [ ]:
# Check file exists
import os
if not os.path.isfile("btc_1h_data.txt"):
    print("Data file not found. Please ensure 'btc_1h_data.txt' is in the current directory. Email me if you want the data file and I forgot to include it (427kb)")


# Load the data you saved earlier
prices = np.loadtxt("btc_1h_data.txt")[:1000]
test_prices = np.loadtxt("btc_1h_data.txt")[:2000]

# Calculate log-prices (this makes the growth linear rather than geometric)
log_data = np.log(prices)

# Start with very small guesses for hourly data
start_mu = 0.0001 
start_sigma = 0.01

mu_btc, sigma_btc, _ = mcmc(prices, dt=1, n_iterations=20000, 
                            start_mu=start_mu, start_sigma=start_sigma, 
                            step_size=0.0005)

# Get the mean estimates after burn-in
est_mu = np.mean(mu_btc[5000:])
est_sigma = np.mean(sigma_btc[5000:])

# Simulate a "Fake BTC" path using your found parameters
_, fake_btc = simulate_gbm(x0=test_prices[0], mu=est_mu, sigma=est_sigma, 
                           T=len(test_prices), dt=1)

plt.figure(figsize=(12, 6))
plt.plot(test_prices, label="Real BTC (Hourly)", alpha=0.8)
plt.plot(fake_btc, label="GBM Model Prediction", alpha=0.8, linestyle='--')
plt.title(f"BTC vs Model (mu={est_mu:.6f}, sigma={est_sigma:.6f})")
plt.legend()
plt.show()

Seems to captre the early trend of the training data and fails when outside. This is expected.

In [ ]:

prices = np.loadtxt("btc_1h_data.txt")
window_size = 400
step_size = 200
n_iterations = 20000  # Lowered for speed during sweep
burn_in = 1000
dt = 1

# Containers
indices = []
mu_means = []
mu_stds = []
sigma_means = []
sigma_stds = []

print(f"Starting Envelope Sweep... (~{len(prices)//step_size} windows)")

# --- Processing ---
for start in range(0, len(prices) - window_size, step_size):
    print(f"Processing window starting at index {start}... [{(start//step_size)+1}/{(len(prices)-window_size)//step_size}]")
    end = start + window_size
    window_data = prices[start:end]
    
    # Run MCMC on the current window
    # Note: Use your existing mcmc/run_mcmc function here
    m_chain, s_chain, _ = mcmc(window_data, dt, 
                                n_iterations=n_iterations, 
                                start_mu=0.0001, 
                                start_sigma=0.005, 
                                step_size=0.0005)
    
    # Analyze post-burn samples
    clean_m = m_chain[burn_in:]
    clean_s = s_chain[burn_in:]
    
    indices.append(end) # Plot at the end of the window
    mu_means.append(np.mean(clean_m))
    mu_stds.append(np.std(clean_m))
    sigma_means.append(np.mean(clean_s))
    sigma_stds.append(np.std(clean_s))



In [ ]:
# Convert to numpy arrays
indices = np.array(indices)
mu_means = np.array(mu_means)
mu_stds = np.array(mu_stds)
sigma_means = np.array(sigma_means)
sigma_stds = np.array(sigma_stds)

# --- Plotting the Envelopes ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# 1. Drift (Mu) Envelope
ax1.plot(indices, mu_means, color='blue', label='Estimated $\mu$ (Mean)')
ax1.fill_between(indices, mu_means - mu_stds, mu_means + mu_stds, 
                 color='blue', alpha=0.2, label='$\mu$ Uncertainty ($\pm 1$ std)')
ax1.axhline(0, color='black', linestyle='--', alpha=0.3)
ax1.set_title('Evolution of Drift ($\mu$) with Uncertainty Envelope')
ax1.set_ylabel('$\mu$ Value')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# 2. Volatility (Sigma) Envelope
ax2.plot(indices, sigma_means, color='green', label='Estimated $\sigma$ (Mean)')
ax2.fill_between(indices, sigma_means - sigma_stds, sigma_means + sigma_stds, 
                 color='green', alpha=0.2, label='$\sigma$ Uncertainty ($\pm 1$ std)')
ax2.set_title('Evolution of Volatility ($\sigma$) with Uncertainty Envelope')
ax2.set_ylabel('$\sigma$ Value')
ax2.set_xlabel('Hours (Time)')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

prices = np.loadtxt("btc_1h_data.txt")
window_size = 800
step_size = 200
n_iterations = 20000  # Lowered for speed during sweep
burn_in = 1000
dt = 1

# Containers
indices = []
mu_means = []
mu_stds = []
sigma_means = []
sigma_stds = []

print(f"Starting Envelope Sweep... (~{len(prices)//step_size} windows)")

# --- Processing ---
for start in range(0, len(prices) - window_size, step_size):
    print(f"Processing window starting at index {start}... [{(start//step_size)+1}/{(len(prices)-window_size)//step_size}]")
    end = start + window_size
    window_data = prices[start:end]
    
    # Run MCMC on the current window
    # Note: Use your existing mcmc/run_mcmc function here
    m_chain, s_chain, _ = mcmc(window_data, dt, 
                                n_iterations=n_iterations, 
                                start_mu=0.0001, 
                                start_sigma=0.005, 
                                step_size=0.0005)
    
    # Analyze post-burn samples
    clean_m = m_chain[burn_in:]
    clean_s = s_chain[burn_in:]
    
    indices.append(end) # Plot at the end of the window
    mu_means.append(np.mean(clean_m))
    mu_stds.append(np.std(clean_m))
    sigma_means.append(np.mean(clean_s))
    sigma_stds.append(np.std(clean_s))

In [ ]:
# Convert to numpy arrays
indices = np.array(indices)
mu_means = np.array(mu_means)
mu_stds = np.array(mu_stds)
sigma_means = np.array(sigma_means)
sigma_stds = np.array(sigma_stds)

# --- Plotting the Envelopes ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# 1. Drift (Mu) Envelope
ax1.plot(indices, mu_means, color='blue', label='Estimated $\mu$ (Mean)')
ax1.fill_between(indices, mu_means - mu_stds, mu_means + mu_stds, 
                 color='blue', alpha=0.2, label='$\mu$ Uncertainty ($\pm 1$ std)')
ax1.axhline(0, color='black', linestyle='--', alpha=0.3)
ax1.set_title('Evolution of Drift ($\mu$) with Uncertainty Envelope')
ax1.set_ylabel('$\mu$ Value')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# 2. Volatility (Sigma) Envelope
ax2.plot(indices, sigma_means, color='green', label='Estimated $\sigma$ (Mean)')
ax2.fill_between(indices, sigma_means - sigma_stds, sigma_means + sigma_stds, 
                 color='green', alpha=0.2, label='$\sigma$ Uncertainty ($\pm 1$ std)')
ax2.set_title('Evolution of Volatility ($\sigma$) with Uncertainty Envelope')
ax2.set_ylabel('$\sigma$ Value')
ax2.set_xlabel('Hours (Time)')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Results comment : 

While Task 1 allowed for a successful identification of parameters for a simulated stationary process, applying the MCMC framework to Bitcoin data reveals the fundamental limitations of the GBM model when confronted with real-world, time evolving, financial data. The results from the sliding-window analysis above (400h vs 800h time window, on a 1h interval) demonstrates that the assumption of constant parameters is invalid, as both drift $\mu$ and volatility $\sigma$ exhibit extreme non-stationarity, shifting significantly across different market regimes. Specifically, the volatility envelopes show clear evidence of "clustering," where periods of relative calm are punctuated by violent, high-stress bursts that a single, global estimate would completely overlook. By comparing the 400-hour and 800-hour windows, we show a critical trade-off in characterization: while the shorter window is more reactive to sudden shifts, it suffers from high drift uncertainty. Conversely, the longer window should provide us with a better accuracy on the drift, however, due to the rather unpredicable nature of financial data, it does not appear to be the case here. In the end, I find that the GBM model is misspecified for Bitcoin, as the asset's nature itself ensures that the underlying parameters are as dynamic as the price action itself, suggesting that a stationary SDE can only ever provide a fleeting, localized snapshot of a fundamentally evolving system. This is however a well known result that I wanted to verify by myself.